In [ ]:
# ── Notebook environment setup ───────────────────────────────────────────
import sys, os
from pathlib import Path

# Locate project root: look for config.py in cwd or parent (handles both
# running from notebooks/ and running from project root)
_cwd = Path(os.getcwd())
ROOT = _cwd if (_cwd / "config.py").exists() else _cwd.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import (
    ROOT_DIR, BOOK_DEPTH_DIR, TRADES_DIR, PROCESSED_DIR,
    ALL_PAIRS_CLEANED, ALL_PAIRS_TRADES, ALL_PAIRS_LABELED,
    DEFAULT_PAIRS, DEFAULT_PERIOD,
    ROLL_WINDOW_SHORT, ROLL_WINDOW_REGIME,
    CRASH_HORIZON, CRASH_SIGMA, LABEL_PAIR,
)
print(f"Project root: {ROOT}")
print(f"Data dir:     {BOOK_DEPTH_DIR}")

This file is responsible for fetching latest data from the Binance API

In [4]:
# Imports

import numpy as np
import pandas as pd
import binance.client as client
import requests
import time
import math

print("All libraries imported successfully.")

All libraries imported successfully.


In [11]:
# Constants

TRADING_PAIRS = []


In [2]:
# Pair Selection

while True:
    TRADING_PAIRS = input("Input the trading pairs (comma-separated, e.g. BTCUSDT,ETHUSDT) or 0 for default: ").strip().upper().split(",")
    
    if TRADING_PAIRS[0] == "0":
        TRADING_PAIRS = ["ETHUSDT"]
        break
    if len(TRADING_PAIRS) >  2 :
        print("Invalid trading pair (too many). Please try again.")
        continue
    for i in range(len(TRADING_PAIRS)):
        if len(TRADING_PAIRS[i]) < 6:
            print(f"Invalid trading pair: {TRADING_PAIRS[i]}. Please try again.")
            continue
    if(len(TRADING_PAIRS) <0):
        print("No trading pairs available. Please try again.")
        continue
    break

print(f"Using trading pairs: {TRADING_PAIRS}")

Using trading pairs: ['ETHUSDT', 'BTCUSDT']


In [13]:
# Data Fetching
# Download historical depth range using helper module

from data_collection import book_depth_utils

period_input = input("Enter period (e.g. 7d, 1w, 1m, 1y, or YYYY-MM-DD/YYYY-MM-DD): ").strip()
if not period_input:
    period_input = "7d"

summary = book_depth_utils.download_book_depth_range(TRADING_PAIRS, period_input, out_base=str(BOOK_DEPTH_DIR), pause_seconds=0.15)
print("Download summary:")
print(f"Symbols: {summary['requested_symbols']}")
print(f"Period: {summary['start_date']} to {summary['end_date']} ({summary['days_requested']} days)")
print(f"Downloaded: {summary['downloaded']}")
print(f"Missing (404): {summary['skipped_404']}")
print(f"Errors: {summary['errors']}")
print("Detail sample:")
for line in summary["details"][:10]:
    print(" -", line)



Download summary:
Symbols: ['ETHUSDT', 'BTCUSDT']
Period: 2026-03-15 to 2026-03-21 (7 days)
Downloaded: 12
Missing (404): 2
Errors: 0
Detail sample:
 - ETHUSDT 2026-03-15: downloaded
 - ETHUSDT 2026-03-16: downloaded
 - ETHUSDT 2026-03-17: downloaded
 - ETHUSDT 2026-03-18: downloaded
 - ETHUSDT 2026-03-19: downloaded
 - ETHUSDT 2026-03-20: downloaded
 - ETHUSDT 2026-03-21: missing (not available)
 - BTCUSDT 2026-03-15: downloaded
 - BTCUSDT 2026-03-16: downloaded
 - BTCUSDT 2026-03-17: downloaded


In [ ]:
# Merge book depth daily CSVs â€” one file at a time, never accumulates more than one day in RAM
import glob
from pathlib import Path

for pair in TRADING_PAIRS:
    path = BOOK_DEPTH_DIR / pair
    if not path.is_dir():
        print(f"No bookDepth_data directory for {pair}")
        continue

    all_files = sorted(
        f for f in glob.glob(str(path / "*.csv"))
        if "_merged" not in Path(f).name
    )
    if not all_files:
        print(f"No daily CSV files found for {pair}")
        continue

    out_path = path / f"{pair}_merged.csv"
    first = True
    for f in all_files:
        chunk = pd.read_csv(f)
        chunk.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
        first = False

    print(f"[{pair}] merged {len(all_files)} daily files â†’ {out_path.name}")


In [ ]:
# Download historical trades using helper module

from data_collection import trades_utils

summary = trades_utils.download_trades_range(TRADING_PAIRS, period_input, out_base=str(TRADES_DIR), pause_seconds=0.15)
print("Trade download summary:")
print(f"Symbols: {summary['requested_symbols']}")
print(f"Period: {summary['start_date']} to {summary['end_date']} ({summary['days_requested']} days)")
print(f"Downloaded: {summary['downloaded']}")
print(f"Missing (404): {summary['skipped_404']}")
print(f"Errors: {summary['errors']}")
for line in summary["details"][:10]:
    print(" -", line)


Trade download summary:
Symbols: ['ETHUSDT', 'BTCUSDT']
Period: 2026-03-15 to 2026-03-21 (7 days)
Downloaded: 12
Missing (404): 2
Errors: 0
 - ETHUSDT 2026-03-15: downloaded
 - ETHUSDT 2026-03-16: downloaded
 - ETHUSDT 2026-03-17: downloaded
 - ETHUSDT 2026-03-18: downloaded
 - ETHUSDT 2026-03-19: downloaded
 - ETHUSDT 2026-03-20: downloaded
 - ETHUSDT 2026-03-21: missing (not available)
 - BTCUSDT 2026-03-15: downloaded
 - BTCUSDT 2026-03-16: downloaded
 - BTCUSDT 2026-03-17: downloaded


In [ ]:
# Merge daily trade CSV files â€” one day at a time, delete each file immediately after writing.
# Peak RAM = one day of trades (~270MB for BTC). Scales to any period.
import glob
from pathlib import Path

TRADE_DTYPES = {
    "id": "int64",
    "price": "float64",
    "qty": "float64",
    "quote_qty": "float64",
    "time": "int64",
    "is_buyer_maker": "bool",
}

for pair in TRADING_PAIRS:
    path = TRADES_DIR / pair
    if not path.is_dir():
        print(f"No trades_data directory for {pair}")
        continue

    all_files = sorted(glob.glob(str(path / f"{pair}-trades-*.csv")))
    if not all_files:
        print(f"No daily trade CSVs found for {pair}")
        continue

    out_path = path / f"{pair}_trades_merged.csv"
    first = True
    for f in all_files:
        chunk = pd.read_csv(f, dtype=TRADE_DTYPES)
        chunk.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
        first = False
        Path(f).unlink()  # free disk immediately â€” never hold more than one day at a time

    print(f"[{pair}] merged {len(all_files)} daily files â†’ {out_path.name}")
